# 🤖 Gemma 4 Tech Interviewer — Colab API Server

This notebook loads your fine-tuned Gemma 4 model from Hugging Face and serves it as a FastAPI server exposed via `ngrok`.

## Steps:
1. Run **Cell 1** to install dependencies.
2. Run **Cell 2** to check GPU.
3. Add your `HF_TOKEN`, `NGROK_TOKEN`, and `NGROK_DOMAIN` secrets in the 🔑 key icon on the left sidebar.
   - Get `NGROK_TOKEN` from [dashboard.ngrok.com/authtokens](https://dashboard.ngrok.com/authtokens)
   - Get a free static `NGROK_DOMAIN` from [dashboard.ngrok.com/domains](https://dashboard.ngrok.com/domains)
4. Run **Cell 3** to load the model.
5. Run **Cell 4** to start the API server and get your public URL.
6. Paste the URL into your backend `.env` as `GEMMA_API_URL=<url>`.

In [13]:
# CELL 1: Install dependencies
!pip install -q transformers peft bitsandbytes accelerate fastapi uvicorn pyngrok huggingface_hub
print('Dependencies installed!')

Dependencies installed!
INFO:     41.78.74.12:0 - "GET /health HTTP/1.1" 200 OK
INFO:     41.78.74.12:0 - "GET /health HTTP/1.1" 200 OK
INFO:     41.78.74.12:0 - "GET /health HTTP/1.1" 200 OK
INFO:     41.78.74.12:0 - "POST /generate-question HTTP/1.1" 404 Not Found
INFO:     41.78.74.12:0 - "GET /health HTTP/1.1" 200 OK
[runsync] task=open_mock_interview_session response_preview='Welcome, Waaberi. Thank you for taking the time to complete this mock interview. Our session will assess your skills in frontend development at a senior level. Please answer each question as thoroughl'
INFO:     41.78.74.12:0 - "POST /runsync HTTP/1.1" 200 OK
INFO:     41.78.74.12:0 - "POST /generate-question HTTP/1.1" 404 Not Found
INFO:     41.78.74.12:0 - "POST /generate-question HTTP/1.1" 404 Not Found
INFO:     41.78.74.12:0 - "POST /generate-question HTTP/1.1" 404 Not Found
[runsync] task=ask_technical_question response_preview='That is a set of excellent JavaScript code refactoring questions. Can you i

In [10]:
# CELL 2: Verify GPU
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No GPU found! Go to Runtime -> Change runtime type -> A100 GPU')

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

GPU: Tesla T4
VRAM: 15.6 GB


In [11]:
# CELL 3: Load fine-tuned Gemma model - LIGHTNING AI

import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# Get Hugging Face token from Lightning environment
HF_TOKEN = os.getenv("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN environment variable is not set")

BASE_MODEL_ID = "google/gemma-4-e2b-it"
ADAPTER_ID = "Mohamud24/gemma-4-tech-interviewer"

# Use BF16 if GPU supports it, otherwise FP16
compute_dtype = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype
)

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    ADAPTER_ID,
    token=HF_TOKEN
)

print("Loading base model...")

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map={'': 0},
    torch_dtype=compute_dtype,
    token=HF_TOKEN
)

print("Loading LoRA adapter...")

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_ID,
    token=HF_TOKEN
)

model.eval()

print("✅ Model ready!")

CUDA available: True
GPU: Tesla T4
Loading tokenizer...
Loading base model...


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

Loading LoRA adapter...
✅ Model ready!


In [ ]:
# CELL 4: Start FastAPI server + ngrok tunnel
import json
import re
import uvicorn
from threading import Thread
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from pyngrok import ngrok, conf
import os

NGROK_TOKEN = os.getenv("GEMMA_NGROK_TOKEN")
NGROK_DOMAIN = os.getenv("GEMMA_NGROK_DOMAIN")

conf.get_default().auth_token = NGROK_TOKEN

app = FastAPI(title='Gemma 4 Tech Interviewer API')

class InterviewRequest(BaseModel):
    endpoint: str
    payload: dict

def generate_response(prompt: str, max_tokens: int = 800, temperature: float = 0.7) -> str:
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=temperature > 0,
            temperature=max(temperature, 0.01),
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

def build_prompt(task: str, payload: dict) -> str:
    candidate = payload.get('candidate_name', 'Candidate')
    lang = payload.get('language', 'en')
    specialization = payload.get('specialization', payload.get('jobRole', payload.get('domain', 'technology')))
    difficulty = payload.get('difficulty', 'mid')
    question = payload.get('question', '')
    answer = payload.get('answer', '')
    candidate_experience = payload.get('candidate_experience', '')
    candidate_education = payload.get('candidate_education', '')
    candidate_projects = payload.get('candidate_projects', '')
    candidate_certifications = payload.get('candidate_certifications', '')
    # Previously the backend prepared these but they were dropped before
    # reaching here — the model only ever saw "specialization + difficulty",
    # with no job/resume context to ground a specific question in. That
    # shallow prompt was a large part of why generated questions read as
    # generic even when they were on-topic.
    job_description = payload.get('job_description', '')
    resume_text = payload.get('resume_text', '')
    responsibilities = payload.get('responsibilities', '')
    supporting_skills = payload.get('supporting_skills', '')

    messages = [
        {'role': 'user', 'content': (
            f'task: {task}\n'
            f'candidate_name: {candidate}\n'
            f'language: {lang}\n'
            f'specialization: {specialization}\n'
            f'difficulty: {difficulty}\n'
            + (f'question: {question}\n' if question else '')
            + (f'answer: {answer}\n' if answer else '')
            + (f'candidate_experience: {candidate_experience}\n' if candidate_experience else '')
            + (f'candidate_education: {candidate_education}\n' if candidate_education else '')
            + (f'candidate_projects: {candidate_projects}\n' if candidate_projects else '')
            + (f'candidate_certifications: {candidate_certifications}\n' if candidate_certifications else '')
            + (f'responsibilities: {responsibilities}\n' if responsibilities else '')
            + (f'supporting_skills: {supporting_skills}\n' if supporting_skills else '')
            + (f'job_description: {job_description[:1500]}\n' if job_description else '')
            + (f'resume_excerpt: {resume_text[:1500]}\n' if resume_text else '')
        )}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def build_score_prompt(payload: dict) -> str:
    """Scoring-specific prompt: adds the calibrated 0-100 rubric and asks for
    structured JSON output. The bare 'task: score_candidate_answer' prompt
    (used by build_prompt for other tasks) gives the model no rubric and no
    output schema, which is why answers of very different quality were
    converging on the same mid-range score with no usable feedback.

    Live testing after the rubric was added still showed the same failure —
    an excellent answer, a correct answer, an "I don't know", and an
    off-topic answer ("bananas and pizza") to the same question all scored
    78. Two things changed here on top of the rubric: (1) runsync() now
    calls generate_response with a low temperature for this task instead of
    the freeform 0.7, since a calibrated number is not a creative-writing
    task and high-temperature sampling was likely a big part of why the
    score kept drifting to the same anchor regardless of input; (2) two
    short contrastive worked examples are added below so the model has a
    concrete low-score and high-score reference point instead of only an
    abstract rubric description.
    """
    candidate = payload.get('candidate_name', 'Candidate')
    lang = payload.get('language', 'en')
    specialization = payload.get('specialization', payload.get('jobRole', payload.get('domain', 'technology')))
    difficulty = payload.get('difficulty', 'mid')
    question = payload.get('question', '')
    answer = payload.get('answer', '')

    candidate_experience = payload.get('candidate_experience', '')
    candidate_education = payload.get('candidate_education', '')
    candidate_projects = payload.get('candidate_projects', '')
    candidate_certifications = payload.get('candidate_certifications', '')
    background_lines = []
    if candidate_experience:
        background_lines.append(f'Experience: {candidate_experience}')
    if candidate_education:
        background_lines.append(f'Education: {candidate_education}')
    if candidate_projects:
        background_lines.append(f'Projects: {candidate_projects}')
    if candidate_certifications:
        background_lines.append(f'Certifications: {candidate_certifications}')
    background_block = (
        'CANDIDATE BACKGROUND (from their resume — use ONLY to calibrate what depth of answer to '
        'expect at their level; still score strictly on the content of "answer" below, not on their '
        'resume):\n' + '\n'.join(background_lines) + '\n\n'
        if background_lines else ''
    )

    is_somali = lang.lower() in ('so', 'somali')
    language_directive = (
        'Write "feedback", every item in "strengths" and "improvements", and "suggestedAnswer" '
        'entirely in Somali. Keep English technical terms that have no established Somali '
        'equivalent (e.g. "API", "server", "database") as-is, but every surrounding sentence '
        'must be Somali — never answer in English or mix languages within a sentence.\n\n'
        if is_somali else
        'Write "feedback", every item in "strengths" and "improvements", and "suggestedAnswer" '
        'entirely in English.\n\n'
    )

    worked_examples = (
        'WORKED EXAMPLES (the score must move like this, not stay near one number):\n'
        'Example A — question "What is a REST API?", answer "I don\'t really know, maybe something '
        'with websites?" -> {"score": 8, "feedback": "No substantive explanation of REST or APIs was '
        'given.", "strengths": [], "improvements": ["Explain what REST means", "Describe how HTTP '
        'methods map to operations"], "suggestedAnswer": "A REST API uses HTTP methods (GET, POST, '
        'PUT, DELETE) to let clients read and modify server resources, typically exchanging JSON."}\n'
        'Example B — question "What is a REST API?", answer "It\'s an architectural style for web '
        'services that uses standard HTTP methods like GET, POST, PUT, DELETE on resources identified '
        'by URLs, usually exchanging JSON, and is stateless so each request carries everything the '
        'server needs." -> {"score": 92, "feedback": "Accurate and complete: covers HTTP methods, '
        'resource-based URLs, statelessness, and JSON.", "strengths": ["Correctly names the core HTTP '
        'verbs", "Explains statelessness, a commonly missed point"], "improvements": ["Could mention '
        'status codes"], "suggestedAnswer": "..."}\n'
        'A real answer almost never lands exactly on 78 — commit to wherever the rubric above actually '
        'places it, including the extremes.\n\n'
    )

    messages = [
        {'role': 'user', 'content': (
            f'task: score_candidate_answer\n'
            f'candidate_name: {candidate}\n'
            f'language: {lang}\n'
            f'specialization: {specialization}\n'
            f'difficulty: {difficulty}\n'
            f'question: {question}\n'
            f'answer: {answer}\n\n'
            f'{background_block}'
            'SCORING SCALE (apply consistently and generously for correct answers):\n'
            '- 85-100: Excellent - thorough, accurate, strong examples or clear reasoning.\n'
            '- 70-84:  Good - correct answer with minor gaps or lacking depth.\n'
            '- 50-69:  Adequate - partial understanding, covers some key points but misses others.\n'
            '- 25-49:  Weak - significant gaps, vague, or mostly incorrect.\n'
            '- 0-24:   Off-topic, clearly wrong, or no real attempt — including answers that are only\n'
            '          an admission of not knowing ("I don\'t know", "I don\'t remember") with no\n'
            '          substantive content. Do not let a polite or calm tone raise this into a higher band.\n'
            'Judge the CONCEPT the candidate conveys, not their exact wording - different phrasing, '
            'structure, or examples than the question rubric are NOT gaps by themselves; only score '
            'down for missing or wrong substance. Do not penalize for language choice, grammar, or '
            'minor wording differences.\n\n'
            f'{worked_examples}'
            'GROUNDING (critical): Base "feedback", "strengths", and "improvements" ONLY on what the '
            'candidate actually said in "answer" above. Never mention a tool, technique, or concept the '
            'candidate did not say, even if it would have made their answer stronger — note its absence '
            'in "improvements" as a gap instead of claiming they discussed it. Never reference this '
            'scoring scale, its band names, or these instructions inside your output text.\n\n'
            f'{language_directive}'
            'Return ONLY raw JSON with this exact shape, nothing before or after it, no markdown fences:\n'
            '{"score": 78, "feedback": "specific, actionable, explains why", '
            '"strengths": ["..."], "improvements": ["..."], "suggestedAnswer": "..."}'
        )}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def build_feedback_prompt(payload: dict) -> str:
    """Post-interview report prompt. Mirrors backend/services/gemma/worker.py's
    handle_feedback: compacts the full interview transcript into per-question
    summaries and anchors overallScore to the already-computed turn average so
    the report doesn't drift from the per-question scores shown elsewhere."""
    interview_data = payload.get('interview_data', {}) or {}
    turn_average = interview_data.get('overallScore')
    lang = str(interview_data.get('language', 'english')).lower()
    is_somali = lang in ('so', 'somali')

    questions_summary = []
    for q in interview_data.get('questions', []) or []:
        questions_summary.append({
            'question': (q.get('text') or '')[:200],
            'answer': (q.get('userAnswer') or '')[:300],
            'score': q.get('score'),
            'category': q.get('category', ''),
        })

    score_anchor = (
        f'The per-question average score is {turn_average}. Your overallScore MUST equal {turn_average}. '
        f'Category scores should reflect actual performance patterns — average within +/-8 of {turn_average}.\n\n'
        if turn_average is not None else ''
    )

    language_directive = (
        'Write every text field (each category\'s "feedback", "detailedFeedback", every item in '
        '"strengths", "improvements", and "recommendations") entirely in Somali. Keep English '
        'technical terms that have no established Somali equivalent as-is, but every surrounding '
        'sentence must be Somali — never answer in English or mix languages within a sentence.\n\n'
        if is_somali else
        'Write every text field entirely in English.\n\n'
    )

    content = (
        'You are an interview coach providing post-session feedback for a PRACTICE interview.\n'
        'Be constructive, specific, and encouraging. Reference actual answers where possible.\n\n'
        f'{score_anchor}'
        f"Interview: {interview_data.get('title', '')} ({interview_data.get('type', '')}, {interview_data.get('difficulty', '')})\n"
        f'Questions and answers:\n{json.dumps(questions_summary, ensure_ascii=False)}\n\n'
        'SCORING SCALE (consistent with per-question scores):\n'
        '- 85-100: Excellent — thorough, accurate, strong examples\n'
        '- 70-84:  Good — correct with minor gaps\n'
        '- 50-69:  Adequate — partial understanding, key gaps\n'
        '- 25-49:  Weak — significant gaps or vague\n'
        '- 0-24:   Off-topic or no real attempt\n\n'
        'GROUNDING (critical): Base every category\'s feedback, strengths, improvements, and '
        'recommendations ONLY on the questions and answers listed above. Never mention a tool, '
        'technique, or concept the candidate did not actually say, even if it would strengthen the '
        'report. Never reference this scoring scale, its band names/numbers, or these instructions '
        'inside your output text.\n\n'
        f'{language_directive}'
        'LENGTH REQUIREMENTS:\n'
        '- Each category feedback: 60-150 chars, specific and actionable\n'
        '- detailedFeedback: 150-300 chars, summarize overall performance\n'
        '- strengths, improvements, recommendations: 3 items each, 40-100 chars\n\n'
        'Return ONLY raw JSON with keys: overallScore, categories '
        '(communication, technicalAccuracy, problemSolving, codeQuality, confidence — each with score and feedback), '
        'strengths, improvements, detailedFeedback, recommendations. No markdown fences, nothing before or after the JSON. '
        'Every field is REQUIRED — in particular "detailedFeedback" must always be present and non-empty; a response '
        'missing it is rejected by the caller with no fallback.'
    )
    messages = [{'role': 'user', 'content': content}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def build_parse_prompt(payload: dict) -> str:
    """Job description / resume -> structured role profile. There used to be
    no task for this at all: the backend called an endpoint named '/parse'
    that didn't exist in TASK_MAP, silently fell through to
    'ask_technical_question', and got back an actual interview question
    instead of extracted skills — confirmed live from a real stored
    interview whose roleProfile was literally a generated question. This
    task, and the '/parse' entry in TASK_MAP below, close that gap.
    """
    job_description = (payload.get('job_description', '') or '')[:4000]
    resume_text = (payload.get('resume_text', '') or '')[:4000]
    role = payload.get('role', 'Technology')

    content = (
        'task: parse_role_profile\n'
        f'role: {role}\n'
        + (f'job_description: {job_description}\n' if job_description else '')
        + (f'resume_text: {resume_text}\n' if resume_text else '')
        + '\nExtract a structured profile from the job description and/or resume above. '
        'If a field has no information available, return an empty list or empty string for it — '
        'never invent skills, experience, or education that aren\'t present in the text.\n'
        'Return ONLY raw JSON with this exact shape, nothing before or after it, no markdown fences:\n'
        '{"requiredSkills": ["..."], "preferredSkills": ["..."], "technicalStack": ["..."], '
        '"responsibilities": ["..."], "experienceLevel": "junior|mid|senior|lead|", '
        '"candidateSkills": ["..."], "candidateExperience": ["..."], "candidateEducation": ["..."], '
        '"candidateProjects": ["..."], "candidateCertifications": ["..."]}'
    )
    messages = [{'role': 'user', 'content': content}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

TASK_MAP = {
    '/ask_technical_question': 'ask_technical_question',
    '/score_candidate_answer': 'score_candidate_answer',
    '/open_mock_interview_session': 'open_mock_interview_session',
    '/close_mock_interview_session': 'close_mock_interview_session',
    '/open_hiring_interview_session': 'open_hiring_interview_session',
    '/close_hiring_interview_session': 'close_hiring_interview_session',
    '/feedback': 'feedback',
    '/parse': 'parse',
}

@app.get('/health')
def health():
    return {'status': 'online', 'model': 'Mohamud24/gemma-4-tech-interviewer', 'provider': 'colab'}

@app.post('/runsync')
async def runsync(req: InterviewRequest):
    task_key = req.endpoint
    if task_key not in TASK_MAP:
        raise HTTPException(status_code=404, detail=f'Unknown endpoint: {task_key}')
    task = TASK_MAP[task_key]
    if task == 'score_candidate_answer':
        prompt = build_score_prompt(req.payload)
    elif task == 'feedback':
        prompt = build_feedback_prompt(req.payload)
    elif task == 'parse':
        prompt = build_parse_prompt(req.payload)
    else:
        prompt = build_prompt(task, req.payload)
    # feedback's schema (5 categories + 3x3 lists + detailedFeedback) needs more
    # headroom than the default 800 — that budget was truncating comprehensive
    # feedback responses mid-generation, producing unparseable JSON.
    max_tokens = 1200 if task == 'feedback' else 800
    # Scoring needs a calibrated number, not creative freeform text — live
    # testing showed temperature=0.7 (the same setting used for question
    # generation) producing the same ~78 score regardless of answer quality,
    # including for off-topic nonsense answers. A near-greedy temperature
    # makes the model actually commit to what the rubric says instead of
    # sampling around a comfortable middle value.
    temperature = 0.2 if task == 'score_candidate_answer' else 0.7
    response = generate_response(prompt, max_tokens=max_tokens, temperature=temperature)
    # Lightweight debug visibility into what the model actually returned for
    # scoring, so a run of suspiciously-similar scores can be diagnosed from
    # the Colab cell output without re-running requests manually. This is the
    # model's own evaluation text, not the raw candidate answer.
    print(f'[runsync] task={task} response_preview={response[:200]!r}')
    return {'output': {'response': response, 'task': task}}

# Start uvicorn in background thread to avoid Colab asyncio conflicts
def run_server():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='info')

server_thread = Thread(target=run_server, daemon=True)
server_thread.start()

# Connect ngrok tunnel with static domain
tunnel = ngrok.connect(8000, 'http', domain=NGROK_DOMAIN)
public_url = tunnel.public_url

print(f'\n==========================================')
print(f'YOUR COLAB GEMMA URL IS READY!')
print(f'URL: {public_url}')
print(f'==========================================')
print(f'Paste this into your backend .env file:')
print(f'GEMMA_API_URL={public_url}')
print(f'==========================================')